# Nested Data the Spark Way: explode, Structs, Schema Inference, Star Schemas

## What you'll practice

Spark handles JSON as `struct`s and `array`s; most real work is reshaping them: `explode`/`explode_outer`, `select("customer.*")`, `collect_list`, schema inference and its conflicts, and finally
normalizing nested documents into fact and dimension tables. You'll implement each in plain Python on `datasets/json/orders_nested.json`.

> Function and class names in **Implement** are what the tests look for. Run all tests for this notebook with `pytest tests/pyspark_prep/test_12_02_nested_data_for_spark.py -v` from the lab root. Untouched exercises show as *skipped*.
>
> Solutions: `solutions/12_pyspark_preparation/02_nested_data_for_spark.ipynb`. Don't open it until you've tried.

## Exercise 1 — 🟢 Easy · explode vs explode_outer

**Implement:** `explode(rows: list[dict], column: str, alias: str, outer: bool = False) -> list[dict]`

One output row per element of the list in `column`, with the element under `alias` and the original list column removed. Rows where the list is empty or `None`: dropped by default; with `outer=True`, kept once with `alias = None`.

**Input:** `rows: list[dict], column: str, alias: str, outer: bool = False`

**Expected output:** `list[dict]`

**Constraints:** Standard library only (the last notebook uses the PySpark DataFrame API).

**Example:** the tests are executable examples — read `test_explode` in `tests/pyspark_prep/test_12_02_nested_data_for_spark.py`.

**Check:** `pytest tests/pyspark_prep/test_12_02_nested_data_for_spark.py -k test_explode`

**YOUR CODE ↓**

## Exercise 2 — 🟡 Medium · Flatten structs into columns

**Implement:** `flatten_structs(row: dict, sep: str = '_') -> dict`

Flatten nested **dicts** into prefixed columns (`customer.contact.email` → `customer_contact_email`), like `select(col('customer.contact.email').alias(...))`. Lists are left as-is (they need explode, not flatten). Name collisions → `ValueError`.

**Input:** `row: dict, sep: str = '_'`

**Expected output:** `dict`

**Constraints:** Standard library only (the last notebook uses the PySpark DataFrame API).

**Example:** the tests are executable examples — read `test_flatten_structs` in `tests/pyspark_prep/test_12_02_nested_data_for_spark.py`.

**Check:** `pytest tests/pyspark_prep/test_12_02_nested_data_for_spark.py -k test_flatten_structs`

**YOUR CODE ↓**

## Exercise 3 — 🟡 Medium · collect_list / collect_set

**Implement:** `collect_by(rows: list[dict], key: str, value: str, distinct: bool = False, sort: bool = False) -> dict[object, list]`

Group values by key into lists (`collect_list`) or distinct lists (`collect_set`, preserving first-seen order unless `sort=True`). `None` values are skipped (Spark skips nulls in collect_*).

**Input:** `rows: list[dict], key: str, value: str, distinct: bool = False, sort: bool = False`

**Expected output:** `dict[object, list]`

**Constraints:** Standard library only (the last notebook uses the PySpark DataFrame API).

**Example**

```python
collect_by([{'c': 1, 't': 'b'}, {'c': 1, 't': 'a'}], 'c', 't', sort=True)
# -> {1: ['a', 'b']}
```

**Check:** `pytest tests/pyspark_prep/test_12_02_nested_data_for_spark.py -k test_collect_by`

**YOUR CODE ↓**

## Exercise 4 — 🟠 Hard · Schema inference with type merging

**Implement:** `infer_schema(records: list[dict]) -> dict`

Infer a Spark-like schema across JSON records: scalar types `"boolean"`, `"long"` (int), `"double"` (float), `"string"`, `"null"` (only nulls seen); nested dicts → `{"struct": {...}}` (merged
field-wise across records); lists → `{"array": element_type}` (merging all elements; empty lists contribute `"null"`). Merge rules: `null` + T → T; `long` + `double` → `double`; struct + struct →
merged struct; array + array → array of merged element; anything else conflicting → `"string"`. Fields missing in some records are simply absent there (nullable). Return a dict `{field: type}` with keys sorted at every struct level.

**Input:** `records: list[dict]`

**Expected output:** `dict`

**Constraints:** Standard library only (the last notebook uses the PySpark DataFrame API).

**Example:** the tests are executable examples — read `test_infer_schema, test_infer_schema_real` in `tests/pyspark_prep/test_12_02_nested_data_for_spark.py`.

**Check:** `pytest tests/pyspark_prep/test_12_02_nested_data_for_spark.py -k "test_infer_schema or test_infer_schema_real"`

**YOUR CODE ↓**

## Challenge — 🔴 Very Hard · Normalize nested orders into a star schema · 🧩 Mini-project

**Implement:** `to_star_schema(orders: list[dict]) -> dict[str, list[dict]]`

From `orders_nested.json` build:
- `dim_customer`: one row per customer id: `customer_key` (int surrogate, 1.. in order of first appearance), `customer_id`, `name`, `country`, `email` (the **latest** order's values win)
- `dim_product`: one row per product id: `product_key`, `product_id`, `name`, `category` (`"UNKNOWN"` when null)
- `fact_order_items`: one row per item: `order_id`, `line_no` (1-based), `customer_key`, `product_key`, `order_date` (`YYYY-MM-DD`), `qty`, `price`, `line_total` (2 dp), `payment_status` (`"unknown"` when missing)
- `fact_orders_without_items`: order ids with no items (so nothing silently disappears).
Then assert referential integrity: every fact key exists in its dimension. Orders are processed in `created_at` order for "latest wins".

**Input:** `orders: list[dict]`

**Expected output:** `dict[str, list[dict]]`

**Constraints:** Standard library only (the last notebook uses the PySpark DataFrame API).

**Example:** the tests are executable examples — read `test_to_star_schema, test_to_star_schema_latest_wins` in `tests/pyspark_prep/test_12_02_nested_data_for_spark.py`.

**Check:** `pytest tests/pyspark_prep/test_12_02_nested_data_for_spark.py -k "test_to_star_schema or test_to_star_schema_latest_wins"`

**YOUR CODE ↓**